# VWAP Drift 01 — conditional drift toward VWAP in volatility-engine units

Pre-registration: `discussions/VWAP_Drift/001_claude_vwd01_preregistration.md`. Code: `src/data_engineering/modeling/vwap_drift.py`.

The question behind a directional option toward VWAP: is
$$\mathbb E\big[\,-\operatorname{sign}(D_t)\,(\log C_{t+h}-\log C_t)\;\big|\;|Z_t|,\ \text{vol regime},\ \text{jump}\big] > \text{costs},\qquad Z_t=D_t/\sigma^{\text{engine}}_{t,h}$$
at $h\in\{15, 60, 240\}$ min and to the next 08:00 expiry? Perpetual only; options only if the gates pass.

Environment: `VWD01_ASSETS` (default `BTC,ETH,SOL,XRP`), `VWD01_SYNTHETIC=1` (synthetic smoke run, outputs to a temp dir), `BINANCE_UM_DIR`.

In [ ]:
import os, sys, json, time, subprocess, tempfile, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import vwap_drift as vd

SYNTHETIC = os.getenv('VWD01_SYNTHETIC', '0') == '1'
ASSETS = [a.strip().upper() for a in os.getenv('VWD01_ASSETS', ','.join(vd.ASSETS)).split(',') if a.strip()]
P = vd.Params()
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='vwd01-')) / '01'; P = vd.Params(n_boot=300, n_boot_table=100)
else:
    OUT = ROOT / 'notebooks' / 'VWAP_Drift' / '01'
    BIN = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance'
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_vwap_drift.py'],
                     cwd=str(ROOT), env=env, capture_output=True, text=True)
print(chk.stderr[-200:]); assert chk.returncode == 0, 'tests failed'
FITS = vd.load_engine(ROOT / 'notebooks' / 'Volatility_Dynamics' / '18' / '18_manifest.json')
print(json.dumps(vd.registration(P), indent=1, default=str))

## 1. Origins, engine forecasts and outcomes (all anchors, real and placebo)

In [ ]:
T0 = time.time(); parts, cov = [], []
for k, a in enumerate(ASSETS):
    if SYNTHETIC:
        df = vd.synthetic_klines(300, seed=11 + k, start='2025-09-01', kappa=0.0)
    else:
        df = pd.read_parquet(BIN / f'{a}USDT_um_1m.parquet', columns=['open_time', 'high', 'low', 'close', 'volume', 'quote_volume'])
    A, info = vd.run_asset(df, a, FITS, P); del df
    parts.append(A); cov.append(info); print(a, f'{time.time() - T0:.0f}s', info['origins'], 'origins')
A = pd.concat(parts, ignore_index=True); del parts
cov = pd.DataFrame(cov); display(cov)
display(A.groupby(['kind', 'test', 'h'], observed=True)['n'].sum().unstack('h'))

## 2. Pre-registered decision: DEV selection, TEST gates

In [ ]:
dev = vd.dev_table(A, P).sort_values('dev_lo', ascending=False)
display(dev.head(12).round(3))
cell = vd.select(dev, P)
decision = vd.evaluate(A, cell, P)
print(json.dumps(decision, indent=1, default=float))
print('VERDICT:', decision['verdict'])

## 3. Secondary S1 — high-volatility regime, |Z| ≥ 2 (Ivan's hypothesis), Bonferroni over 8

In [ ]:
s1 = vd.secondary_s1(A, P)
display(s1.round(3))

## 4. Descriptive tables (no decision)

In [ ]:
tz = vd.table(A, ['anchor', 'h', 'zb'], P, seed=11)
tv = vd.table(A[A.zb.isin(['z2_3', 'z3p'])], ['anchor', 'h', 'vol'], P, seed=12)
tj = vd.table(A[A.zb.isin(['z2_3', 'z3p'])], ['anchor', 'h', 'jump'], P, seed=13)
cells = vd.table(A, vd.CELL, P, seed=14)
show = ['mean_bp', 'lo', 'hi', 'placebo_mean_bp', 'diff', 'diff_lo', 'diff_hi', 'hit', 'p0', 'n', 'days']
print('TEST, by |Z| bucket'); display(tz[tz.test].set_index(['anchor', 'h', 'zb'])[show].round(2))
print('TEST, |Z| >= 2, by volatility tercile'); display(tv[tv.test].set_index(['anchor', 'h', 'vol'])[show].round(2))
print('TEST, |Z| >= 2, by recent jump'); display(tj[tj.test].set_index(['anchor', 'h', 'jump'])[show].round(2))

## 5. Figures

In [ ]:
H = list(vd.HORIZONS); ZL = list(vd.ZB)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=False)
for ax, h in zip(axes, H):
    for an, col in (('session', '#1f5aa6'), ('h4', '#c2571a')):
        d = tz[(tz.test) & (tz.anchor == an) & (tz.h == h)].set_index('zb').reindex(ZL)
        x = np.arange(len(ZL)) + (-0.08 if an == 'session' else 0.08)
        ax.errorbar(x, d.mean_bp, yerr=[d.mean_bp - d.lo, d.hi - d.mean_bp], fmt='o-', color=col, ms=4, lw=1.2, capsize=2, label=f'{an} (real)')
        ax.plot(x, d.placebo_mean_bp, ls=':', color=col, lw=1, label=f'{an} placebo')
    ax.axhline(0, color='0.5', lw=0.6); ax.axhline(P.tradeable_bp, color='0.3', lw=0.6, ls='--')
    ax.set_xticks(range(len(ZL))); ax.set_xticklabels(['<1', '1-2', '2-3', '≥3']); ax.set_xlabel('|Z| (engine units)')
    ax.set_title(f'h = {h}' + (' min' if h != 'EXP' else ' (to 08:00)'), loc='left', fontsize=10); ax.grid(alpha=0.25)
axes[0].set_ylabel('drift toward VWAP, bp'); axes[0].legend(fontsize=7, frameon=False)
fig.suptitle('TEST: mean return toward VWAP by displacement (dashed: 7 bp perp cost; 95% day-bootstrap CI)', fontsize=11)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, h in zip(axes, H):
    for an, col in (('session', '#1f5aa6'), ('h4', '#c2571a')):
        d = tv[(tv.test) & (tv.anchor == an) & (tv.h == h)].set_index('vol').reindex(list(vd.VOL))
        x = np.arange(3) + (-0.08 if an == 'session' else 0.08)
        ax.errorbar(x, d.mean_bp, yerr=[d.mean_bp - d.lo, d.hi - d.mean_bp], fmt='o-', color=col, ms=4, lw=1.2, capsize=2, label=f'{an} (real)')
        ax.plot(x, d.placebo_mean_bp, ls=':', color=col, lw=1, label=f'{an} placebo')
    ax.axhline(0, color='0.5', lw=0.6); ax.axhline(P.tradeable_bp, color='0.3', lw=0.6, ls='--')
    ax.set_xticks(range(3)); ax.set_xticklabels(vd.VOL); ax.set_xlabel('engine volatility tercile')
    ax.set_title(f'h = {h}' + (' min' if h != 'EXP' else ' (to 08:00)'), loc='left', fontsize=10); ax.grid(alpha=0.25)
axes[0].set_ylabel('drift toward VWAP, bp'); axes[0].legend(fontsize=7, frameon=False)
fig.suptitle('TEST, |Z| ≥ 2: does a high-volatility regime bring price back to VWAP?', fontsize=11)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
d = tz[tz.test]
for h, mk in zip(H, 'osD^'):
    g = d[d.h == h]; ax.scatter(g.p0, g.hit, marker=mk, s=22, label=h)
ax.plot([0, 1], [0, 1], color='k', lw=0.7)
ax.set_xlabel('driftless reflection benchmark p0'); ax.set_ylabel('observed hit rate of the VWAP level')
ax.legend(title='h', fontsize=8, frameon=False); ax.set_title('Does price reach VWAP more often than a random walk would?', fontsize=10)
plt.tight_layout(); plt.show()

## 6. Exports

In [ ]:
dev.to_csv(OUT / '01_dev_selection.csv', index=False, float_format='%.6g')
s1.to_csv(OUT / '01_secondary_s1.csv', index=False, float_format='%.6g')
tz.to_csv(OUT / '01_by_zbucket.csv', index=False, float_format='%.6g')
tv.to_csv(OUT / '01_by_vol_regime.csv', index=False, float_format='%.6g')
tj.to_csv(OUT / '01_by_jump.csv', index=False, float_format='%.6g')
cells.to_csv(OUT / '01_all_cells.csv', index=False, float_format='%.6g')
cov.to_csv(OUT / '01_coverage.csv', index=False)
(OUT / '01_decision.json').write_text(json.dumps({'decision': decision, 'registration': vd.registration(P)}, indent=1, default=float))
(OUT / '01_manifest.json').write_text(json.dumps({'module': 'VWAP_Drift_01', 'version': vd.VERSION, 'synthetic': SYNTHETIC,
    'assets': ASSETS, 'origins_total': int(A.n.sum()), 'runtime_s': round(time.time() - T0, 1)}, indent=1))
print(sorted(p.name for p in OUT.iterdir()))